# POLAR H10 BELT DATA STREAM

## Loads and imports

### Packages

In [1]:
# Imports
# For bluetooth streaming
import asyncio
from bleak import BleakScanner      # This is for scanning available devices
from bleak import BleakClient       # This is for streaming

# Unix timestamping
import time
from datetime import datetime, timezone

# Data handling and data viz
import pandas as pd
import plotly.express as px

# Display real time measurements
from IPython.display import display

This is not something I understand 100%. The device address is clear. It's the device's unique identifier. The **HR_CHAR** (HR Measurement Characteristic) is something that should either show 180D (for Heart Rate Service, or 2a37 for heart rate measurement characteristic. It's in the first chunk's last few digits: 0000xxxx). But I probably don't really need to understand this perfectly, just be aware that these are heart rate sensor specific stuff.

### Setups

In [2]:
DEVICE_ADDRESS = "FFDB0E1C-0262-9016-D154-4562DABCBE43"
HR_CHAR = "00002a37-0000-1000-8000-00805f9b34fb"
HEART_RATE_SERVICE_UUID = "0000180d-0000-1000-8000-00805f9b34fb"

stream_seconds = 60 # This is the default measurement length is seconds. Can be overwritten in params.

## Looking for devices

In [3]:
def format_bytes(data: bytes) -> str:
    return data.hex(" ") if data else ""

async def scan_ble_devices(timeout: float = 8.0):
    results = await BleakScanner.discover(timeout=timeout, return_adv=True)

    if not results:
        print("No BLE devices found.")
        return {}

    print(f"Found {len(results)} BLE device(s).\n")
    for address, (device, adv) in results.items():
        display_name = adv.local_name or device.name or "<no name in advertisement>"
        print(f"Address:     {address}")
        print(f"Name:        {display_name}")
        print(f"RSSI:        {adv.rssi} dBm")       # Technically, signal strength.

        if adv.service_uuids:
            print("Service UUIDs:")
            for uuid in adv.service_uuids:
                marker = "  [Heart Rate Service]" if uuid.lower() == HEART_RATE_SERVICE_UUID else ""
                print(f"  - {uuid}{marker}")

        if adv.manufacturer_data:
            print("Manufacturer data:")
            for company_id, payload in adv.manufacturer_data.items():
                print(f"  - company_id=0x{company_id:04X}, data={format_bytes(payload)}")

        if adv.service_data:
            print("Service data:")
            for uuid, payload in adv.service_data.items():
                print(f"  - {uuid}: {format_bytes(payload)}")

        if adv.tx_power is not None:
            print(f"TX power:    {adv.tx_power} dBm")

        print("-" * 60)

    return results

scan_results = await scan_ble_devices(timeout=8.0)

Found 17 BLE device(s).

Address:     861B13E2-FEF3-CA33-3CD1-3E0CAE7761B0
Name:        <no name in advertisement>
RSSI:        -60 dBm
Service data:
  - 0000fef3-0000-1000-8000-00805f9b34fb: 4a 17 23 58 46 33 37 11 32 fe 81 5b 61 c7 6b db 5a e6 1c 84 79 0e 2e e4 b2 44 d4
------------------------------------------------------------
Address:     FCDC2FF3-C2FA-BC30-8BC4-7D4258E4275D
Name:        01GYA2320C0325
RSSI:        -96 dBm
Service UUIDs:
  - 6e400001-b5a3-f393-e0a9-e50e24dcca9e
Manufacturer data:
  - company_id=0x424E, data=83 02 01 00 00 79
------------------------------------------------------------
Address:     943FB2AC-A0DC-8C2C-0371-6E947841BEEC
Name:        <no name in advertisement>
RSSI:        -57 dBm
Manufacturer data:
  - company_id=0x004C, data=10 06 34 1e 88 3c 22 04 01 00 00 00 00 00 00 00 00 00 00 00 00 80 00 00 00
TX power:    12 dBm
------------------------------------------------------------
Address:     8B24B9C7-EEC3-F20F-0D46-9F9851F4CE3F
Name:        <no name

## Connection test

In [4]:

device_address_to_check = DEVICE_ADDRESS
HEART_RATE_SERVICE_UUID = "0000180d-0000-1000-8000-00805f9b34fb"

async def check_device_access(device_address: str, timeout: float = 10.0):
    print(f"Looking for device {device_address}...")
    device = await BleakScanner.find_device_by_address(device_address, timeout=timeout)
    if device is None:
        print("Device not found. Make sure the belt is awake and advertising.")
        return False

    print(f"Found: {device.name} ({device.address})")
    print("Connecting...")

    async with BleakClient(device) as client:
        print(f"Connected: {client.is_connected}")

        if hasattr(client, "get_services"):
            services = await client.get_services()
        else:
            services = client.services

        if services is None:
            print("No services discovered.")
            return False

        service_list = list(services)
        print(f"Service count: {len(service_list)}")

        has_hr_service = any(
            s.uuid.lower() == HEART_RATE_SERVICE_UUID for s in service_list
        )
        print(f"Heart Rate service present: {has_hr_service}")

        print("\nGATT services and characteristics:")
        for service in service_list:
            print(f"[Service] {service.uuid}")
            for char in service.characteristics:
                props = ", ".join(char.properties)
                print(f"  - {char.uuid} ({props})")

    print("Disconnected.")
    return True

connected_ok = await check_device_access(device_address_to_check)
connected_ok

Looking for device FFDB0E1C-0262-9016-D154-4562DABCBE43...
Found: Polar H10 E6B05E24 (FFDB0E1C-0262-9016-D154-4562DABCBE43)
Connecting...
Connected: True
Service count: 6
Heart Rate service present: True

GATT services and characteristics:
[Service] 0000180d-0000-1000-8000-00805f9b34fb
  - 00002a37-0000-1000-8000-00805f9b34fb (notify)
  - 00002a38-0000-1000-8000-00805f9b34fb (read)
[Service] 0000180a-0000-1000-8000-00805f9b34fb
  - 00002a29-0000-1000-8000-00805f9b34fb (read)
  - 00002a24-0000-1000-8000-00805f9b34fb (read)
  - 00002a25-0000-1000-8000-00805f9b34fb (read)
  - 00002a27-0000-1000-8000-00805f9b34fb (read)
  - 00002a26-0000-1000-8000-00805f9b34fb (read)
  - 00002a28-0000-1000-8000-00805f9b34fb (read)
  - 00002a23-0000-1000-8000-00805f9b34fb (read)
[Service] 0000180f-0000-1000-8000-00805f9b34fb
  - 00002a19-0000-1000-8000-00805f9b34fb (read, notify)
[Service] 6217ff4b-fb31-1140-ad5a-a45545d7ecf3
  - 6217ff4c-c8ec-b1fb-1380-3ad986708e2d (read)
  - 6217ff4d-91bb-91d0-7e2a-7cd3bd

True

## Streaming Heart Rate Data

In the byte array, the first one [0] is always some flag, the second [1] is the actual heart rate. The rest is all kinds of stuff. This was the same on the Decathlon device as well. I guess this is the standard.

### Predefining functions

In [5]:
def parse_hr_measurement(data: bytearray) -> int:
    return data[1]  # The heart rate is the 2nd byte in the array

async def collect_hr(stream_seconds=stream_seconds):
    t_values = []
    hr_values = []

    live_handle = display("Waiting for connection...", display_id=True)

    def handle_hr(sender, data):
        hr = parse_hr_measurement(data)
        t_values.append(time.time())
        hr_values.append(hr)
        live_handle.update(f"HR: {hr} bpm")

    async with BleakClient(DEVICE_ADDRESS) as client:
        await client.start_notify(HR_CHAR, handle_hr)
        await asyncio.sleep(stream_seconds)
        await client.stop_notify(HR_CHAR)

        print("Data collection ended.")

    return pd.DataFrame({"timestamp": t_values, "hr": hr_values})

### Streaming data

In [ ]:
df = await collect_hr(120)

'HR: 82 bpm'

Data collection ended.


In [7]:
df.to_csv("hrtest.csv", index=False)

## EDA

In [12]:
data = df.copy()
data.head()

,timestamp,hr
0,1.773849e+09,78
1,1.773849e+09,78
2,1.773849e+09,79
3,1.773849e+09,79
4,1.773849e+09,79


In [14]:
fig = px.line(data, x = "timestamp", y = "hr", title = "R peaks over time")
fig.show()

## SANDBOX

In [5]:
data = pd.read_csv("hrtest.csv")

In [6]:
tmp = 1284101485
tmp = int(1.773847e+09)
print(datetime.fromtimestamp(tmp, tz=timezone.utc))

2026-03-18 15:16:40+00:00


In [8]:
unix_tss = data["timestamp"].to_list()
unix_tss

[1773848803.988103,
 1773848804.976962,
 1773848805.983117,
 1773848806.987401,
 1773848807.978052,
 1773848808.982191,
 1773848809.987572,
 1773848810.977584,
 1773848811.982653,
 1773848812.9876869,
 1773848813.977961,
 1773848814.98293,
 1773848815.987217,
 1773848816.977587,
 1773848817.981795,
 1773848818.987614,
 1773848819.977563,
 1773848820.9818828,
 1773848821.988153,
 1773848822.977194,
 1773848823.982632,
 1773848824.988158,
 1773848825.977476,
 1773848826.98292,
 1773848827.987309,
 1773848828.97717,
 1773848829.982919,
 1773848830.9865642,
 1773848831.981722,
 1773848833.593608,
 1773848834.027771,
 1773848835.333123,
 1773848836.20278,
 1773848837.072201,
 1773848838.814442,
 1773848839.248108,
 1773848840.116479,
 1773848840.992971,
 1773848842.2964349,
 1773848843.167569,
 1773848844.039118,
 1773848845.3427148,
 1773848846.212778,
 1773848847.0831392,
 1773848848.388211,
 1773848849.257664,
 1773848850.127616,
 1773848850.9977012,
 1773848852.3036668,
 1773848853.1730

In [9]:
def unixtime_converter(unix_timestamp):
    dt = datetime.fromtimestamp(unix_timestamp, tz = timezone.utc)
    print(dt)

unixtime_converter(tmp)

2026-03-18 15:16:40+00:00


In [15]:
unixtime_converter(unix_tss[0])

2026-03-18 15:46:43.988103+00:00


In [26]:
converted_tss = [unixtime_converter(x) for x in unix_tss if True]
converted_tss[0]

2026-03-18 15:46:43.988103+00:00
2026-03-18 15:46:44.976962+00:00
2026-03-18 15:46:45.983117+00:00
2026-03-18 15:46:46.987401+00:00
2026-03-18 15:46:47.978052+00:00
2026-03-18 15:46:48.982191+00:00
2026-03-18 15:46:49.987572+00:00
2026-03-18 15:46:50.977584+00:00
2026-03-18 15:46:51.982653+00:00
2026-03-18 15:46:52.987687+00:00
2026-03-18 15:46:53.977961+00:00
2026-03-18 15:46:54.982930+00:00
2026-03-18 15:46:55.987217+00:00
2026-03-18 15:46:56.977587+00:00
2026-03-18 15:46:57.981795+00:00
2026-03-18 15:46:58.987614+00:00
2026-03-18 15:46:59.977563+00:00
2026-03-18 15:47:00.981883+00:00
2026-03-18 15:47:01.988153+00:00
2026-03-18 15:47:02.977194+00:00
2026-03-18 15:47:03.982632+00:00
2026-03-18 15:47:04.988158+00:00
2026-03-18 15:47:05.977476+00:00
2026-03-18 15:47:06.982920+00:00
2026-03-18 15:47:07.987309+00:00
2026-03-18 15:47:08.977170+00:00
2026-03-18 15:47:09.982919+00:00
2026-03-18 15:47:10.986564+00:00
2026-03-18 15:47:11.981722+00:00
2026-03-18 15:47:13.593608+00:00
2026-03-18